# LeetCode 3347 — Maximum Frequency of an Element After Performing Operations II bằng Python

- Link: https://leetcode.com/problems/maximum-frequency-of-an-element-after-performing-operations-ii/
- Độ khó: **Hard**
- Chủ đề: **Sắp xếp + cửa sổ trượt / hai con trỏ**, **sweep line (mảng hiệu trên tọa độ thưa)**, tách hai trường hợp **giá trị đích có sẵn / không có sẵn** trong mảng

Skeleton trên LeetCode (Python3):

```python
class Solution:
    def maxFrequency(self, nums: List[int], k: int, numOperations: int) -> int:
```

## 1. Mục tiêu học

1. Đổi góc nhìn: thay vì chọn phép biến đổi, **chọn giá trị đích `T`** rồi hỏi "tối đa bao nhiêu phần tử thành `T`?".
2. Với `T` cố định: phần tử `x` thành được `T` ⇔ `|x − T| ≤ k`; phần tử **đã bằng `T`** không tốn thao tác ⇒ tần suất `= min(W(T), freq(T) + numOperations)`, `W(T)` = số phần tử trong `[T − k, T + k]`.
3. Chỉ cần thử hữu hạn `T`: các giá trị **có sẵn** và các điểm **`x − k`** (hoặc `x + k`).
4. Cài đặt `O(n log n)` bằng: bisect trên ứng viên, sweep line, hoặc **hai con trỏ** trên mảng đã sắp xếp.
5. Kiểm chứng bằng vét cạn mọi `T` trên giá trị nhỏ.

## 2. Đề bài & ví dụ

Cho `nums`, `k`, `numOperations`. Thực hiện **đúng** `numOperations` thao tác; mỗi thao tác chọn một chỉ số **chưa từng được chọn** và cộng vào nó một số nguyên trong `[−k, k]`. Trả về **tần suất lớn nhất** có thể của một phần tử sau các thao tác.

**Ví dụ 1:** `nums = [1,4,5]`, `k = 1`, `numOperations = 2` → `2` (`[1,4,4]`).
**Ví dụ 2:** `nums = [5,11,20,20]`, `k = 5`, `numOperations = 1` → `2`.

**Ràng buộc:** `1 ≤ n ≤ 10^5`; `1 ≤ nums[i] ≤ 10^9`; `0 ≤ k ≤ 10^9`; `0 ≤ numOperations ≤ n`.

> "Đúng `numOperations` lần" không gây khó: thao tác thừa có thể **cộng 0** vào một chỉ số khác (hoặc vào chính phần tử đã bằng `T`).

In [1]:
from typing import List
from collections import Counter
import bisect

examples = [([1, 4, 5], 1, 2, 2), ([5, 11, 20, 20], 5, 1, 2)]
for nums, k, ops, exp in examples:
    print(nums, 'k =', k, 'ops =', ops, '-> kỳ vọng', exp)

[1, 4, 5] k = 1 ops = 2 -> kỳ vọng 2
[5, 11, 20, 20] k = 5 ops = 1 -> kỳ vọng 2


## 3. Kiến thức nền

### 3.1. Đếm phần tử trong đoạn bằng bisect

Trên mảng đã sắp `a`, số phần tử trong `[L, R]` là `bisect_right(a, R) − bisect_left(a, L)`.

In [2]:
a = sorted([5, 11, 20, 20])
def count_in(a, L, R):
    return bisect.bisect_right(a, R) - bisect.bisect_left(a, L)
print('trong [15, 25]:', count_in(a, 15, 25), '| trong [6, 16]:', count_in(a, 6, 16))

trong [15, 25]: 2 | trong [6, 16]: 1


### 3.2. Tần suất đạt được cho một đích `T`

- `W(T)` = số phần tử trong `[T − k, T + k]` — các phần tử **có thể** thành `T`.
- Trong đó `freq(T)` phần tử **đã** bằng `T` (không tốn thao tác), còn lại mỗi phần tử tốn 1 thao tác.
- ⇒ `best(T) = min(W(T), freq(T) + numOperations)`.

### 3.3. Ứng viên `T` nào là đủ?

- Nếu `freq(T) > 0`: `T` là một giá trị có sẵn — thử mọi giá trị có sẵn.
- Nếu `freq(T) = 0`: `best(T) = min(W(T), ops)`; `W(T)` là hàm bậc thang theo `T`, chỉ tăng tại các điểm `x − k` ⇒ giá trị lớn nhất đạt tại một điểm `x − k`. Tương đương: số phần tử lớn nhất trong một **cửa sổ độ rộng `2k`**.

In [3]:
def best_of(nums, k, ops, T):
    c = Counter(nums)
    W = sum(1 for x in nums if abs(x - T) <= k)
    return min(W, c[T] + ops)

nums, k, ops = [1, 4, 5], 1, 2
print({T: best_of(nums, k, ops, T) for T in range(0, 7)})

{0: 1, 1: 1, 2: 1, 3: 1, 4: 2, 5: 2, 6: 1}


## 4. Ý tưởng nền tảng / cách nghĩ trực giác

Ví dụ 2: `[5, 11, 20, 20]`, `k = 5`, `ops = 1`.

- `T = 20`: `W = |{20, 20}| = 2` (`11` cách 9 > 5), `freq = 2` ⇒ `min(2, 3) = 2`.
- `T = 16` (không có sẵn): `W = |{11, 20, 20}| = 3` nhưng `freq = 0` ⇒ `min(3, 1) = 1`.
- `T = 11`: đoạn `[6, 16]` chỉ chứa `11` ⇒ `W = 1` ⇒ `1`.
- `T = 10` (không có sẵn): `[5, 15]` chứa `5, 11` ⇒ `min(2, 0 + 1) = 1`.

⇒ đáp án `2`. Bài học: **phần tử đã bằng đích rất "rẻ"**, nên đích có sẵn thường thắng khi `ops` nhỏ; khi `ops` lớn, đích ở giữa một cụm dày (không cần có sẵn) lại thắng (ví dụ `[1, 3]`, `k = 1`, `ops = 2` ⇒ `T = 2` cho `2`).

## 5. Cách 1 — Brute Force: thử mọi `T`

In [4]:
def brute(nums, k, ops):
    c = Counter(nums)
    best = 0
    for T in range(min(nums) - k, max(nums) + k + 1):
        W = sum(1 for x in nums if abs(x - T) <= k)
        best = max(best, min(W, c[T] + ops))
    return best

for nums, k, ops, exp in examples:
    print(brute(nums, k, ops), '| kỳ vọng', exp)

2 | kỳ vọng 2
2 | kỳ vọng 2


**Độ phức tạp:** `O((max − min + 2k) · n)` — chỉ dùng khi giá trị nhỏ.

## 6. Ý tưởng thuật toán / bước đột phá

Sắp xếp `a`. Hai phần:

1. **Đích có sẵn** `x` (duyệt các giá trị phân biệt tăng dần): hai con trỏ `l`, `r` giữ cửa sổ `a[l..r)` = các phần tử trong `[x − k, x + k]` (cả hai biên đều tăng theo `x`) ⇒ `min(r − l, freq(x) + ops)`.
2. **Đích tùy ý**: cửa sổ trượt lớn nhất với `a[r] − a[l] ≤ 2k` ⇒ `min(r − l + 1, ops)`.

Đáp án là max hai phần. Tổng `O(n log n)` (do sắp xếp).

## 7. Các cách tiếp theo

### 7.1. Cách 2 — Ứng viên `{x − k, x, x + k}` + bisect

In [5]:
def candidates(nums, k, ops):
    a = sorted(nums)
    c = Counter(a)
    best = 0
    for x in c:
        for T in (x - k, x, x + k):
            W = bisect.bisect_right(a, T + k) - bisect.bisect_left(a, T - k)
            best = max(best, min(W, c[T] + ops))
    return best

for nums, k, ops, exp in examples:
    print(candidates(nums, k, ops), '| kỳ vọng', exp)

2 | kỳ vọng 2
2 | kỳ vọng 2


Ở đây ta thử cả `x + k` cho đối xứng (không bắt buộc), mỗi ứng viên `O(log n)`.

### 7.2. Cách 3 — Sweep line trên tọa độ thưa

Mỗi phần tử `x` "phủ" đoạn đích `[x − k, x + k]`: sự kiện `+1` tại `x − k`, `−1` tại `x + k + 1`. Duyệt các tọa độ quan trọng (điểm sự kiện **và** các giá trị có sẵn) tăng dần, giữ `cur = W(T)`.

In [6]:
def sweep(nums, k, ops):
    c = Counter(nums)
    diff = Counter()
    for x in nums:
        diff[x - k] += 1
        diff[x + k + 1] -= 1
    keys = sorted(set(diff) | set(c))
    cur = best = 0
    for T in keys:
        cur += diff[T]
        best = max(best, min(cur, c[T] + ops))
    return best

for nums, k, ops, exp in examples:
    print(sweep(nums, k, ops), '| kỳ vọng', exp)

2 | kỳ vọng 2
2 | kỳ vọng 2


**Lưu ý:** phải thêm các giá trị có sẵn vào `keys` — tại `T = x` có thể không có sự kiện nào nhưng `freq(T)` lớn.

### 7.3. Cách 4 — Hai con trỏ (lời giải chính)

In [7]:
def two_pointers(nums, k, ops):
    a = sorted(nums)
    n = len(a)
    c = Counter(a)
    best = 0
    l = r = 0
    for x in sorted(c):                      # đích có sẵn
        while a[l] < x - k:
            l += 1
        while r < n and a[r] <= x + k:
            r += 1
        best = max(best, min(r - l, c[x] + ops))
    l = 0
    for r in range(n):                       # đích tùy ý: cửa sổ độ rộng 2k
        while a[r] - a[l] > 2 * k:
            l += 1
        best = max(best, min(r - l + 1, ops))
    return best

for nums, k, ops, exp in examples:
    print(two_pointers(nums, k, ops), '| kỳ vọng', exp)

2 | kỳ vọng 2
2 | kỳ vọng 2


## 8. TRACE từng bước

Trace cách 4 trên `nums = [1, 2, 4, 4, 7, 8, 9]`, `k = 2`, `ops = 2`.

In [8]:
def trace(nums, k, ops):
    a = sorted(nums); n = len(a); c = Counter(a)
    print('a =', a)
    best = 0; l = r = 0
    for x in sorted(c):
        while a[l] < x - k: l += 1
        while r < n and a[r] <= x + k: r += 1
        val = min(r - l, c[x] + ops)
        best = max(best, val)
        print(f"  đích có sẵn {x}: cửa sổ {a[l:r]} (W = {r - l}), freq = {c[x]} -> min({r - l}, {c[x] + ops}) = {val}")
    l = 0
    for r in range(n):
        while a[r] - a[l] > 2 * k: l += 1
        val = min(r - l + 1, ops)
        best = max(best, val)
        print(f"  đích tùy ý, cửa sổ {a[l:r + 1]} (rộng {a[r] - a[l]} <= {2 * k}) -> min({r - l + 1}, {ops}) = {val}")
    print('đáp án:', best, '| vét cạn:', brute(nums, k, ops))

trace([1, 2, 4, 4, 7, 8, 9], 2, 2)

a = [1, 2, 4, 4, 7, 8, 9]
  đích có sẵn 1: cửa sổ [1, 2] (W = 2), freq = 1 -> min(2, 3) = 2
  đích có sẵn 2: cửa sổ [1, 2, 4, 4] (W = 4), freq = 1 -> min(4, 3) = 3
  đích có sẵn 4: cửa sổ [2, 4, 4] (W = 3), freq = 2 -> min(3, 4) = 3
  đích có sẵn 7: cửa sổ [7, 8, 9] (W = 3), freq = 1 -> min(3, 3) = 3
  đích có sẵn 8: cửa sổ [7, 8, 9] (W = 3), freq = 1 -> min(3, 3) = 3
  đích có sẵn 9: cửa sổ [7, 8, 9] (W = 3), freq = 1 -> min(3, 3) = 3
  đích tùy ý, cửa sổ [1] (rộng 0 <= 4) -> min(1, 2) = 1
  đích tùy ý, cửa sổ [1, 2] (rộng 1 <= 4) -> min(2, 2) = 2
  đích tùy ý, cửa sổ [1, 2, 4] (rộng 3 <= 4) -> min(3, 2) = 2
  đích tùy ý, cửa sổ [1, 2, 4, 4] (rộng 3 <= 4) -> min(4, 2) = 2
  đích tùy ý, cửa sổ [4, 4, 7] (rộng 3 <= 4) -> min(3, 2) = 2
  đích tùy ý, cửa sổ [4, 4, 7, 8] (rộng 4 <= 4) -> min(4, 2) = 2
  đích tùy ý, cửa sổ [7, 8, 9] (rộng 2 <= 4) -> min(3, 2) = 2
đáp án: 3 | vét cạn: 3


## 9. Code nộp LeetCode

In [9]:
class Solution:
    def maxFrequency(self, nums: List[int], k: int, numOperations: int) -> int:
        a = sorted(nums)
        n = len(a)
        best = 0
        # Phần 1: đích là giá trị có sẵn
        l = r = 0
        i = 0
        while i < n:
            x = a[i]
            j = i
            while j < n and a[j] == x:
                j += 1
            f = j - i                         # tần suất của x
            while a[l] < x - k:
                l += 1
            if r < j:
                r = j
            while r < n and a[r] <= x + k:
                r += 1
            cand = r - l
            if cand > f + numOperations:
                cand = f + numOperations
            if cand > best:
                best = cand
            i = j
        # Phần 2: đích tùy ý (cửa sổ độ rộng 2k)
        if numOperations > best:
            l = 0
            two_k = 2 * k
            for r in range(n):
                while a[r] - a[l] > two_k:
                    l += 1
                cand = r - l + 1
                if cand > numOperations:
                    cand = numOperations
                if cand > best:
                    best = cand
        return best

for nums, k, ops, exp in examples:
    got = Solution().maxFrequency(nums, k, ops)
    print(got, 'OK' if got == exp else 'SAI')

2 OK
2 OK


**Tối ưu nhỏ:** phần 2 cho kết quả `≤ numOperations`, nên nếu `best ≥ numOperations` thì bỏ qua phần 2.

## 10. Vì sao thuật toán đúng

1. **Đủ thao tác ⇒ đạt `best(T)`:** chọn `min(W(T), freq(T) + ops)` phần tử trong `[T − k, T + k]`, ưu tiên các phần tử đã bằng `T`; mỗi phần tử còn lại dùng 1 thao tác cộng `T − x ∈ [−k, k]`. Thao tác thừa cộng 0 cho chỉ số khác.
2. **Không thể hơn `best(T)`:** phần tử ngoài `[T − k, T + k]` không thành `T` được; phần tử khác `T` cần ít nhất 1 thao tác, mỗi chỉ số chỉ được chọn 1 lần.
3. **Ứng viên đủ:** nếu `freq(T) > 0` thì `T` có sẵn (phần 1). Nếu `freq(T) = 0` thì `best(T) = min(W(T), ops)`; các phần tử trong `[T − k, T + k]` nằm trong một cửa sổ độ rộng `2k`, và ngược lại mọi cửa sổ `[a_l, a_r]` với `a_r − a_l ≤ 2k` được phủ bởi `T = a_l + k` (phần 2 cho cận dưới đúng vì khi `T` trùng giá trị có sẵn, `freq(T) + ops ≥ ops`).
4. **Hai con trỏ:** cả `x − k` và `x + k` tăng theo `x` ⇒ `l`, `r` chỉ tiến.

## 11. Phân tích Time / Space Complexity

| Cách | Time | Space |
|---|---|---|
| 1. Thử mọi `T` | `O((V + 2k) · n)` | `O(n)` |
| 2. Ứng viên + bisect | `O(n log n)` | `O(n)` |
| 3. Sweep line | `O(n log n)` | `O(n)` |
| 4. Hai con trỏ | `O(n log n)` (sắp xếp) + `O(n)` | `O(n)` |

## 12. Bộ test

### 12.1. Ví dụ đề + ca biên

In [10]:
impls = {'candidates': candidates, 'sweep': sweep, 'two_pointers': two_pointers,
         'submit': lambda a, b, c: Solution().maxFrequency(a, b, c)}
edge = [
    ([5], 0, 0, 1),
    ([1, 1, 1], 0, 0, 3),
    ([1, 3], 1, 2, 2),          # đích 2 không có sẵn
    ([1, 2, 3], 1, 1, 2),
    ([1, 10], 100, 0, 1),        # không có thao tác
    ([1, 10], 100, 1, 2),
    ([2, 2, 9, 9, 9], 3, 2, 3),
]
for nums, k, ops, exp in examples + edge:
    assert brute(nums, k, ops) == exp, (nums, k, ops, brute(nums, k, ops))
    for name, f in impls.items():
        assert f(nums, k, ops) == exp, (name, nums, k, ops)
print('Ví dụ đề và ca biên: tất cả đúng.')

Ví dụ đề và ca biên: tất cả đúng.


### 12.2. Đối chiếu ngẫu nhiên với vét cạn

In [11]:
import random
rng = random.Random(3347)
cnt = 0
for _ in range(1500):
    n = rng.randint(1, 8)
    nums = [rng.randint(1, 20) for _ in range(n)]
    k = rng.randint(0, 5)
    ops = rng.randint(0, n)
    exp = brute(nums, k, ops)
    for name, f in impls.items():
        assert f(nums, k, ops) == exp, (name, nums, k, ops)
    cnt += 1
print('Khớp vét cạn trên', cnt, 'bộ test ngẫu nhiên.')

Khớp vét cạn trên 1500 bộ test ngẫu nhiên.


### 12.3. Ca lớn `n = 10^5`

In [12]:
import time
N = 10 ** 5
cases = {
    'ngẫu nhiên, k = 10^6': ([rng.randint(1, 10 ** 9) for _ in range(N)], 10 ** 6, N // 3),
    'nhiều trùng lặp': ([rng.randint(1, 100) for _ in range(N)], 3, 500),
    'k = 0': ([rng.randint(1, 1000) for _ in range(N)], 0, N),
    'k khổng lồ': ([rng.randint(1, 10 ** 9) for _ in range(N)], 10 ** 9, N - 1),
}
for name, (nums, k, ops) in cases.items():
    t = time.time(); a = Solution().maxFrequency(nums, k, ops); t1 = time.time() - t
    t = time.time(); b = sweep(nums, k, ops); t2 = time.time() - t
    t = time.time(); c = candidates(nums, k, ops); t3 = time.time() - t
    assert a == b == c
    print(f"{name:22s}: {a:6d} | nộp {t1:.2f}s | sweep {t2:.2f}s | bisect {t3:.2f}s")

ngẫu nhiên, k = 10^6  :    261 | nộp 0.08s | sweep 0.37s | bisect 0.22s
nhiều trùng lặp       :   1599 | nộp 0.02s | sweep 0.02s | bisect 0.02s
k = 0                 :    137 | nộp 0.04s | sweep 0.03s | bisect 0.02s


k khổng lồ            : 100000 | nộp 0.05s | sweep 0.40s | bisect 0.24s


Kiểm tra độc lập: với `k` khổng lồ mọi phần tử đều với tới mọi đích ⇒ đáp án `= min(n, max_freq + ops)`; với `k = 0` không phần tử nào đổi được ⇒ đáp án `= max_freq`.

In [13]:
nums, k, ops = cases['k khổng lồ']
assert Solution().maxFrequency(nums, k, ops) == min(N, max(Counter(nums).values()) + ops)
nums, k, ops = cases['k = 0']
assert Solution().maxFrequency(nums, k, ops) == max(Counter(nums).values())
print('k khổng lồ và k = 0: khớp công thức.')

k khổng lồ và k = 0: khớp công thức.


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Chỉ thử đích là giá trị có sẵn

In [14]:
def wrong_existing_only(nums, k, ops):
    a = sorted(nums); c = Counter(a)
    return max(min(bisect.bisect_right(a, x + k) - bisect.bisect_left(a, x - k), c[x] + ops) for x in c)

print('sai :', wrong_existing_only([1, 3], 1, 2), '| đúng:', Solution().maxFrequency([1, 3], 1, 2))

sai : 1 | đúng: 2


**Vì sao sai:** đích `2` không có trong mảng nhưng cả `1` và `3` đều với tới (`±1`).

### Lỗi 2 — Quên rằng phần tử đã bằng đích không tốn thao tác: `min(W, ops)`

In [15]:
def wrong_cap_ops(nums, k, ops):
    a = sorted(nums); best = 0; l = 0
    for r in range(len(a)):
        while a[r] - a[l] > 2 * k:
            l += 1
        best = max(best, min(r - l + 1, ops))          # SAI cho đích có sẵn
    return best

print('sai :', wrong_cap_ops([5, 11, 20, 20], 5, 1), '| đúng:', Solution().maxFrequency([5, 11, 20, 20], 5, 1))

sai : 1 | đúng: 2


**Vì sao sai:** hai số `20` đã bằng nhau — tần suất `2` không cần thao tác nào.

### Lỗi 3 — Cửa sổ độ rộng `k` thay vì `2k` cho đích tùy ý

In [16]:
def wrong_width_k(nums, k, ops):
    a = sorted(nums); c = Counter(a); best = 0
    for x in c:
        best = max(best, min(bisect.bisect_right(a, x + k) - bisect.bisect_left(a, x - k), c[x] + ops))
    l = 0
    for r in range(len(a)):
        while a[r] - a[l] > k:                          # SAI: hai phần tử cách nhau tới 2k vẫn gặp nhau ở giữa
            l += 1
        best = max(best, min(r - l + 1, ops))
    return best

bad = ([1, 5], 2, 2)
print('sai :', wrong_width_k(*bad), '| đúng:', Solution().maxFrequency(*bad))

sai : 1 | đúng: 2


**Vì sao sai:** `1 + 2 = 3 = 5 − 2` — hai phần tử cách nhau `2k = 4` vẫn gặp nhau tại `T = 3`.

### Lỗi 4 — Không chặn trên bởi số thao tác

In [17]:
def wrong_no_cap(nums, k, ops):
    a = sorted(nums); best = 0; l = 0
    for r in range(len(a)):
        while a[r] - a[l] > 2 * k:
            l += 1
        best = max(best, r - l + 1)                     # SAI: thiếu min(·, freq + ops)
    return best

print('sai :', wrong_no_cap([1, 2, 3], 1, 1), '| đúng:', Solution().maxFrequency([1, 2, 3], 1, 1))

sai : 3 | đúng: 2


**Vì sao sai:** muốn cả ba thành `2` cần đổi `1` và `3` — 2 thao tác, nhưng chỉ có 1.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Tần suất lớn nhất sau khi mỗi phần tử dịch tối đa `k`" ⇒ cố định đích, đếm trong đoạn `[T − k, T + k]`.
- Hàm đếm theo `T` là bậc thang ⇒ chỉ cần thử tại các điểm gãy (`x ± k`) và các giá trị có sẵn.
- Phân biệt **"đã đúng sẵn" (miễn phí)** và **"cần sửa" (tốn ngân sách)** — mẫu quen thuộc.

## 15. Biến thể của bài

1. **LC 3346 (bản I):** giá trị `≤ 10^5` ⇒ mảng hiệu trên toàn trục.
2. **LC 2831/1838 họ "Frequency of the Most Frequent Element":** thao tác `+1` có chi phí, ngân sách tổng.
3. Mỗi chỉ số được chọn **nhiều lần** ⇒ thực chất dịch tối đa `numOperations · k`… (cẩn thận đề).
4. Trả về **mọi** giá trị đích đạt tần suất lớn nhất (dùng sweep line ghi lại đoạn).

In [18]:
# Biến thể 1: mảng hiệu trên toàn trục (giá trị nhỏ)
def max_freq_small_values(nums, k, ops):
    M = max(nums) + k + 2
    diff = [0] * (M + 1)
    c = [0] * (M + 1)
    for x in nums:
        diff[max(0, x - k)] += 1
        diff[x + k + 1] -= 1
        c[x] += 1
    cur = best = 0
    for T in range(M):
        cur += diff[T]
        best = max(best, min(cur, c[T] + ops))
    return best

rng = random.Random(15)
for _ in range(500):
    nums = [rng.randint(1, 30) for _ in range(rng.randint(1, 8))]
    k = rng.randint(0, 6); ops = rng.randint(0, len(nums))
    assert max_freq_small_values(nums, k, ops) == brute(nums, k, ops)
print('Biến thể mảng hiệu toàn trục khớp vét cạn.')

Biến thể mảng hiệu toàn trục khớp vét cạn.


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Ghi chú |
|---|---|---|---|
| 1 | Thử mọi `T` | `O((V + 2k)n)` | Oracle |
| 2 | Ứng viên + bisect | `O(n log n)` | Dễ viết nhất |
| 3 | Sweep line thưa | `O(n log n)` | Tổng quát |
| 4 | Hai con trỏ | `O(n log n)` | **Nộp bài**, hằng số nhỏ |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output):** `nums = [1, 2, 3, 4]`, `k = 1`, `ops = 4`.

**Bài 2 (tính tay):** `nums = [3, 3, 3, 10]`, `k = 3`, `ops = 0`.

**Bài 3 (điền TODO):** trả về luôn **một giá trị đích** đạt tần suất lớn nhất.

**Bài 4 (tự viết lại):** mỗi phần tử `i` có `k_i` riêng (dịch tối đa `k_i`).

**Bài 5 (mini challenge):** đếm **số giá trị đích nguyên** đạt tần suất lớn nhất.

In [19]:
def argbest_todo(nums, k, ops):
    # TODO
    raise NotImplementedError

def individual_k_todo(nums, ks, ops):
    # TODO
    raise NotImplementedError

def count_best_targets_todo(nums, k, ops):
    # TODO
    raise NotImplementedError

print('Đã định nghĩa các hàm TODO (không gọi).')

Đã định nghĩa các hàm TODO (không gọi).


## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** (1) Đích `2` hoặc `3`. (2) `W = 3`. (3) `3`.

**Bài 2.** (1) Không có thao tác. (2) Tần suất có sẵn. (3) `3`.

**Bài 3.** (1) Dùng sweep line (cách 3). (2) Ghi lại `T` khi cập nhật `best`. (3) Kiểm bằng `best_of`.

**Bài 4.** (1) Phần tử `i` phủ đoạn `[x_i − k_i, x_i + k_i]`. (2) Sweep line vẫn dùng được. (3) Thêm giá trị có sẵn vào tập tọa độ.

**Bài 5.** (1) `best(T)` hằng số trên mỗi đoạn giữa hai tọa độ quan trọng liên tiếp — trừ tại giá trị có sẵn. (2) Tách điểm có sẵn ra riêng. (3) Cộng độ dài các đoạn đạt max.

## 19. Đáp án tham khảo cho bài tập

In [20]:
# Bài 1, 2, 3
assert Solution().maxFrequency([1, 2, 3, 4], 1, 4) == 3 == brute([1, 2, 3, 4], 1, 4)
assert Solution().maxFrequency([3, 3, 3, 10], 3, 0) == 3

def argbest(nums, k, ops):
    c = Counter(nums); diff = Counter()
    for x in nums:
        diff[x - k] += 1; diff[x + k + 1] -= 1
    cur = best = 0; arg = None
    for T in sorted(set(diff) | set(c)):
        cur += diff[T]
        v = min(cur, c[T] + ops)
        if v > best:
            best, arg = v, T
    return best, arg

rng = random.Random(3)
for _ in range(400):
    nums = [rng.randint(1, 20) for _ in range(rng.randint(1, 8))]
    k = rng.randint(0, 5); ops = rng.randint(0, len(nums))
    b, T = argbest(nums, k, ops)
    assert b == brute(nums, k, ops) == best_of(nums, k, ops, T)
print('Bài 1: 3 | Bài 2: 3 | Bài 3: đích trả về luôn đạt tối ưu.')

Bài 1: 3 | Bài 2: 3 | Bài 3: đích trả về luôn đạt tối ưu.


In [21]:
# Bài 4, 5
def individual_k(nums, ks, ops):
    c = Counter(nums); diff = Counter()
    for x, kk in zip(nums, ks):
        diff[x - kk] += 1; diff[x + kk + 1] -= 1
    cur = best = 0
    for T in sorted(set(diff) | set(c)):
        cur += diff[T]
        best = max(best, min(cur, c[T] + ops))
    return best

def count_best_targets(nums, k, ops):
    c = Counter(nums); diff = Counter()
    for x in nums:
        diff[x - k] += 1; diff[x + k + 1] -= 1
    keys = sorted(set(diff) | set(c))
    best, cur, segs = 0, 0, []
    for idx, T in enumerate(keys):
        cur += diff[T]
        segs.append((T, cur))
    vals = []                                  # (giá trị, số điểm nguyên)
    for idx, (T, w) in enumerate(segs):
        vals.append((min(w, c[T] + ops), 1))  # chính điểm T
        if idx + 1 < len(segs):
            gap = segs[idx + 1][0] - T - 1     # các điểm (T, T_next) không có sẵn, W = w
            if gap > 0:
                vals.append((min(w, ops), gap))
    best = max(v for v, _ in vals)
    return sum(cnt for v, cnt in vals if v == best)

def count_best_brute(nums, k, ops):
    vals = {T: best_of(nums, k, ops, T) for T in range(min(nums) - k - 1, max(nums) + k + 2)}
    m = max(vals.values())
    return sum(1 for v in vals.values() if v == m)

rng = random.Random(4)
for _ in range(400):
    nums = [rng.randint(1, 15) for _ in range(rng.randint(1, 7))]
    k = rng.randint(0, 4); ops = rng.randint(1, len(nums))
    ks = [rng.randint(0, 4) for _ in nums]
    exp = 0
    cc = Counter(nums)
    for T in range(-5, 25):
        W = sum(1 for x, kk in zip(nums, ks) if abs(x - T) <= kk)
        exp = max(exp, min(W, cc[T] + ops))
    assert individual_k(nums, ks, ops) == exp
    assert count_best_targets(nums, k, ops) == count_best_brute(nums, k, ops), (nums, k, ops)
print('Bài 4, 5 khớp vét cạn.')

Bài 4, 5 khớp vét cạn.


Ghi chú bài 5: với `ops ≥ 1`, tần suất lớn nhất `≥ 1` nên các điểm ngoài vùng phủ (`W = 0`) không bao giờ đạt max — vùng vét cạn `[min − k − 1, max + k + 1]` là đủ.

## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

```python
a = sorted(nums)
# đích có sẵn x: hai con trỏ giữ a trong [x-k, x+k]  -> min(W, freq(x) + ops)
# đích tùy ý: cửa sổ a[r] - a[l] <= 2k                -> min(len, ops)
return max(...)
```

**Checklist:**

- [ ] Đã thử cả đích **không có sẵn** (cửa sổ độ rộng `2k`)?
- [ ] Phần tử đã bằng đích được miễn thao tác (`freq + ops`)?
- [ ] Chặn trên bởi `ops` cho đích tùy ý?
- [ ] `k = 0`, `ops = 0` cho đáp án là tần suất có sẵn lớn nhất?
- [ ] Sắp xếp trước khi dùng hai con trỏ?